# Colorado Freight Flow Analysis, 2018–2024
### What's driving the decline in Colorado's freight tonnage? An analysis of commodity and mode shifts

**Data source:** FAF5.7.1 Regional Database 2018–2024, Bureau of Transportation Statistics / FHWA
**Analyst:** Jake Chervitz
**Framework:** Ask → Prepare → Process → Analyze → Share → Act

---
## Process: Data Cleaning & Preparation

This notebook filters the ~2M-row national FAF5.7.1 regional file down to Colorado-origin
domestic flows, decodes numeric codes to readable labels, reshapes to long (tidy) format,
and exports a clean analysis-ready dataset.

**Cleaning changelog** (update as you go):
1. Filtered to Colorado origin zones (081 Denver-Aurora, 089 Rest of CO)
2. Filtered to domestic flows only (trade_type == 1)
3. Dropped foreign-trade columns (fr_orig, fr_dest, fr_inmode, fr_outmode), dist_band,
   current_value_* (nominal dollars — using constant 2017 dollars instead), and tmiles_*
4. Decoded dms_mode and sctg2 to descriptive labels per FAF5 data dictionary
5. Reshaped year columns to long format (one row per flow-year)


In [9]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.1f}'.format)

## 1. Load the raw file

The full regional CSV is ~2M rows. We only need a small slice, so we load with
`usecols` to keep memory down — a deliberate big-file technique worth mentioning
in your write-up.

In [10]:
# TODO: update path to wherever you unzipped the file
RAW_PATH = 'FAF5.7.1_2018-2024.csv'

keep_cols = (
    ['dms_orig', 'dms_dest', 'dms_mode', 'sctg2', 'trade_type']
    + [f'tons_{y}' for y in range(2018, 2025)]
    + [f'value_{y}' for y in range(2018, 2025)]
)

df_raw = pd.read_csv(RAW_PATH, usecols=keep_cols)
print(f"Rows loaded: {len(df_raw):,}")
df_raw.head()

Rows loaded: 2,494,901


,dms_orig,dms_dest,dms_mode,sctg2,trade_type,tons_2018,tons_2019,tons_2020,tons_2021,tons_2022,tons_2023,tons_2024,value_2018,value_2019,value_2020,value_2021,value_2022,value_2023,value_2024
0,11,11,1,1,1,51.9,54.0,54.4,56.1,56.4,55.9,56.0,68.6,71.4,71.9,74.2,74.6,73.9,74.1
1,11,19,1,1,1,392.1,408.3,411.2,424.2,426.6,422.6,423.3,518.6,540.0,543.9,561.1,564.3,558.9,559.8
2,11,129,1,1,1,1.4,1.4,1.5,1.5,1.5,1.5,1.5,1.8,1.9,1.9,2.0,2.0,2.0,2.0
3,11,131,1,1,1,12.7,13.2,13.3,13.7,13.8,13.7,13.7,16.8,17.5,17.6,18.2,18.3,18.1,18.1
4,11,139,1,1,1,5.2,5.4,5.5,5.6,5.7,5.6,5.6,6.9,7.2,7.2,7.5,7.5,7.4,7.5


## 2. Filter to the study scope

- **Origin:** FAF zones 081 (Denver-Aurora CO) and 089 (Rest of CO)
- **Trade type:** 1 = domestic flows only

*Decision note:* imports/exports (trade_type 2 and 3) are excluded so the analysis
reflects domestically-originated freight; foreign flows attribute tonnage differently
and would muddy the mode story.

In [11]:
CO_ZONES = [81, 89]

df = df_raw[
    (df_raw['dms_orig'].isin(CO_ZONES)) &
    (df_raw['trade_type'] == 1)
].copy()

df = df.drop(columns=['trade_type'])

print(f"Rows after filtering: {len(df):,}")
print(f"Origin zones present: {sorted(df['dms_orig'].unique())}")

Rows after filtering: 8,600
Origin zones present: [np.int64(81), np.int64(89)]


## 3. Decode numeric codes to labels

Mappings from the FAF5 data dictionary (faf.ornl.gov → Documentation).
Mode and zone maps are complete; the SCTG map covers all 42 commodity groups.

In [12]:
mode_map = {
    1: 'Truck',
    2: 'Rail',
    3: 'Water',
    4: 'Air (incl. truck-air)',
    5: 'Multiple modes & mail',
    6: 'Pipeline',
    7: 'Other and unknown',
    8: 'No domestic mode',
}

zone_map = {81: 'Denver-Aurora CO', 89: 'Rest of CO'}

sctg_map = {
    1: 'Live animals/fish', 2: 'Cereal grains', 3: 'Other ag products',
    4: 'Animal feed', 5: 'Meat/seafood', 6: 'Milled grain products',
    7: 'Other foodstuffs', 8: 'Alcoholic beverages', 9: 'Tobacco products',
    10: 'Building stone', 11: 'Natural sands', 12: 'Gravel',
    13: 'Nonmetallic minerals', 14: 'Metallic ores', 15: 'Coal',
    16: 'Crude petroleum', 17: 'Gasoline', 18: 'Fuel oils',
    19: 'Natural gas and other fossil products', 20: 'Basic chemicals',
    21: 'Pharmaceuticals', 22: 'Fertilizers', 23: 'Chemical products',
    24: 'Plastics/rubber', 25: 'Logs', 26: 'Wood products',
    27: 'Newsprint/paper', 28: 'Paper articles', 29: 'Printed products',
    30: 'Textiles/leather', 31: 'Nonmetal mineral products',
    32: 'Base metals', 33: 'Articles of base metal', 34: 'Machinery',
    35: 'Electronics', 36: 'Motorized vehicles', 37: 'Transport equipment',
    38: 'Precision instruments', 39: 'Furniture', 40: 'Misc. manufactured products',
    41: 'Waste/scrap', 42: 'Unknown', 43: 'Mixed freight',
}

df['origin']    = df['dms_orig'].map(zone_map)
df['mode']      = df['dms_mode'].map(mode_map)
df['commodity'] = df['sctg2'].map(sctg_map)

# Energy commodity flag — central to the analysis
ENERGY_SCTG = [15, 16, 17, 18, 19]
df['energy_group'] = np.where(df['sctg2'].isin(ENERGY_SCTG),
                              'Energy (SCTG 15-19)', 'All other freight')

# Validate: no unmapped codes
assert df['mode'].notna().all(), 'Unmapped mode codes present'
assert df['commodity'].notna().all(), 'Unmapped SCTG codes present'
df[['origin', 'mode', 'commodity', 'energy_group']].head()

,origin,mode,commodity,energy_group
266,Denver-Aurora CO,Truck,Live animals/fish,All other freight
267,Denver-Aurora CO,Truck,Live animals/fish,All other freight
268,Denver-Aurora CO,Truck,Live animals/fish,All other freight
269,Denver-Aurora CO,Truck,Live animals/fish,All other freight
270,Denver-Aurora CO,Truck,Live animals/fish,All other freight


## 4. Reshape to long (tidy) format

Wide year columns → one row per flow-year, with separate `tons` and `value` measures.
Tidy format makes the groupby/plotting code in Analyze dramatically simpler.

In [13]:
id_vars = ['origin', 'dms_dest', 'mode', 'commodity', 'sctg2', 'energy_group']

tons = df.melt(id_vars=id_vars,
               value_vars=[f'tons_{y}' for y in range(2018, 2025)],
               var_name='year', value_name='ktons')
tons['year'] = tons['year'].str.replace('tons_', '').astype(int)

value = df.melt(id_vars=id_vars,
                value_vars=[f'value_{y}' for y in range(2018, 2025)],
                var_name='year', value_name='value_m2017usd')
value['year'] = value['year'].str.replace('value_', '').astype(int)

tidy = tons.merge(value, on=id_vars + ['year'], how='left')
print(f"Tidy rows: {len(tidy):,}")
tidy.head()

Tidy rows: 60,200


,origin,dms_dest,mode,commodity,sctg2,energy_group,year,ktons,value_m2017usd
0,Denver-Aurora CO,81,Truck,Live animals/fish,1,All other freight,2018,0.0,0.0
1,Denver-Aurora CO,89,Truck,Live animals/fish,1,All other freight,2018,85.0,496.6
2,Denver-Aurora CO,209,Truck,Live animals/fish,1,All other freight,2018,8.4,48.9
3,Denver-Aurora CO,319,Truck,Live animals/fish,1,All other freight,2018,142.9,835.3
4,Denver-Aurora CO,350,Truck,Live animals/fish,1,All other freight,2018,0.3,1.7


## 5. Validation checks

Sanity-check the cleaned data against known totals from the FAF Data Tabulation Tool
recon (Pass 1). Denver 2018 outbound was ~156,170 ktons total across modes
(from the tabulation tool, all trade types) — domestic-only will be somewhat lower.
The point is order-of-magnitude agreement.

In [14]:
# Totals by origin and year
check = tidy.groupby(['origin', 'year'])['ktons'].sum().unstack()
check

year,2018,2019,2020,2021,2022,2023,2024
origin,,,,,,,
Denver-Aurora CO,"154,228.7","155,529.1","148,547.0","143,564.3","144,482.3","145,262.0","137,786.8"
Rest of CO,"143,330.9","140,059.4","123,174.7","124,754.2","119,505.9","123,384.0","112,671.3"


In [15]:
# Quick sniff test: 2018 -> 2024 change by mode
mode_check = (tidy.groupby(['mode', 'year'])['ktons'].sum().unstack()
              .assign(pct_chg=lambda d: (d[2024] / d[2018] - 1) * 100)
              .sort_values(2018, ascending=False))
mode_check

year,2018,2019,2020,2021,2022,2023,2024,pct_chg
mode,,,,,,,,
Truck,"174,749.8","178,740.1","168,920.0","171,140.2","166,291.7","168,542.7","157,921.9",-9.6
Pipeline,"97,112.3","92,094.0","82,554.8","75,194.4","75,296.1","77,920.0","71,219.5",-26.7
Rail,"21,525.4","20,422.3","16,337.8","17,893.4","18,558.8","18,202.0","17,476.4",-18.8
Multiple modes & mail,"4,141.2","4,301.9","3,881.7","4,062.2","3,812.8","3,952.7","3,812.0",-7.9
Air (incl. truck-air),30.9,30.2,27.5,28.4,28.7,28.6,28.3,-8.4


## 6. Export analysis-ready dataset

In [16]:
OUT_PATH = 'co_freight_tidy_2018_2024.csv'
tidy.to_csv(OUT_PATH, index=False)
print(f"Saved {len(tidy):,} rows to {OUT_PATH}")

Saved 60,200 rows to co_freight_tidy_2018_2024.csv


---
## Next: Analyze

Five charts, one per sub-question:
1. Total CO outbound tonnage by mode, 2018–2024 (line)
2. Absolute tonnage change by commodity, 2018 → 2024 (bar — 'the money chart')
3. Energy vs. all other freight, indexed 2018 = 100 (line)
4. Energy commodities individually: coal, crude, gasoline, fuel oils, nat. gas products (small multiples)
5. Truck-only stability view + top stable commodities (the 'so what' chart)

*Palette note: consider reusing the navy/coral/cream/sage portfolio palette for brand consistency.*

In [17]:
tidy['mode'].unique()

array(['Truck', 'Multiple modes & mail', 'Rail', 'Air (incl. truck-air)',
       'Pipeline'], dtype=object)